In [1]:
!pip install graphviz # In a Jupyter cell
from graphviz import Digraph
import math
import numpy as np
import matplotlib.pyplot as plt
%matplotlib

Using matplotlib backend: module://matplotlib_inline.backend_inline


In [18]:
class Value:
    def __init__(self, data, children = (), op = "", label = ""):
        self.data = data
        self.op = op
        self.label = label

        self.children = set(children)
        self.grad = 0
        self._backward = lambda: None
        
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")

        def _backward():
            self.grad += out.grad;
            other.grad += out.grad;
    
        out._backward = _backward

        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")
        
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
            
        out._backward = _backward

        return out

    def __pow__(self, other):
        if not isinstance(other, (int, float)):
            raise RuntimeError("pow only supports ints and floats")
            
        out = Value(self.data ** other, (self,), "**")

        def _backward():
            # power rule
            self.grad += other * (self.data ** (other-1)) * out.grad;

        out._backward = _backward

        return out

    def exp(self):
        v = math.exp(self.data)
        out = Value(v, (self,), "exp")

        def _backward():
            self.grad += v * out.grad

        out._backward = _backward

        return out

    def tanh(self):
        # (e ^ 2x - 1) / (e ^ 2x + 1)
        x = self.data
        v = (math.exp(2 * x) - 1) / (math.exp(2 * x) + 1)
        
        out = Value(v, (self,), "tanh")

        def _backward(): 
            self.grad += (1. - v ** 2) * out.grad
            
        out._backward = _backward

        return out

    def backward(self): 
        self.grad = 1.0

        visited = set()
        topo = []
        
        def build_topo(node):
            if node in visited:
                return
        
            visited.add(node)
            
            for child in node.children:
                build_topo(child)
            
            topo.append(node)
        
        build_topo(self)

        for node in reversed(topo):
            node._backward()

    def __truediv__(self, other):
        return self * (other **-1)

    def __radd__(self, other):
        return self + other
    
    def __rtruediv__(self, other): # other / self
        return other * self**-1
        
    def __rmul__(self, other):
        return self * other

    def __neg__(self):
        return self * -1

    def __sub__(self, other):
        return self + (-other)

    def __repr__(self):
        return f"Value(data = {self.data}, grad = {self.grad})"


In [3]:
def trace(root):
    nodes, edges = set(), set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v.children:
                edges.add((child, v))
                build(child)
    build(root)
    return nodes, edges

def draw_dot(root, format='svg', rankdir='LR'):
    """
    format: png | svg | ...
    rankdir: TB (top to bottom graph) | LR (left to right)
    """
    assert rankdir in ['LR', 'TB']
    nodes, edges = trace(root)
    dot = Digraph(format=format, graph_attr={'rankdir': rankdir}) #, node_attr={'rankdir': 'TB'})
    
    for n in nodes:
        dot.node(name=str(id(n)), label = "{ %s | data %.4f | grad %.4f }" % (n.label, n.data, n.grad), shape='record')
        if n.op:
            dot.node(name=str(id(n)) + n.op, label=n.op)
            dot.edge(str(id(n)) + n.op, str(id(n)))
    
    for n1, n2 in edges:
        dot.edge(str(id(n1)), str(id(n2)) + n2.op)
    
    return dot

In [19]:
import random

class Neuron:
    def __init__(self, dim):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(0, dim)]
        self.b = Value(random.uniform(-1, 1))

    def __call__(self, x):
        r = sum(((wi * xi) for wi, xi in zip(self.w, x)), self.b)
        return r.tanh()

    def parameters(self):
        return self.w + [self.b]

class Layer:
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(0, nout)]

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs

    def parameters(self):
        params = []
        for neuron in self.neurons:
            params.extend(neuron.parameters())
        return params

class MLP:
    def __init__(self, nin, nouts): # nin is a int, nouts is a list of ints
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        params = []
        for layer in self.layers:
            params.extend(layer.parameters())
        return params
        

In [105]:
mlp = MLP(3, [4,4,1])

mlp([1,2,3])

Value(data = 0.1787033505638962, grad = 0)

In [106]:
xs = [
    [2.0,3.0,1.0],
    [3.0,-1, .5],
    [.5, 1.0, 1.0],
    [1.0,1.0,-1.0],
]
ys = [1.0, -1.0, -1.0, 1.0]

In [107]:
h = 0.01

its = 0
maxits = 100000

minloss = 0.0001

while True:
    #
    ypred = [mlp(x) for x in xs]
    loss = sum([(pi - yi) ** 2 for (yi, pi) in zip(ys, ypred)])

    for p in mlp.parameters():
        p.grad = 0
    
    loss.backward()

    for p in mlp.parameters():
        p.data -= h * p.grad

    its += 1
    
    if its > maxits:
        break;

    if loss.data < minloss:
        break;

print(loss.data)

9.999750252042645e-05


In [104]:
[mlp(x) for x in xs]

[Value(data = 0.9951340991985126, grad = 0),
 Value(data = -0.9944644366781573, grad = 0),
 Value(data = -0.994814547660692, grad = 0),
 Value(data = 0.9956668941719511, grad = 0)]